In [0]:
print("calalog and schema setup")

spark.sql("Use catalog workspace")
spark.sql("Use SCHEMA default")

print("Current catalog:",spark.sql("Select current_catalog()").first()[0])

print("Current schema:",spark.sql("Select current_schema()").first()[0])

calalog and schema setup
Current catalog: workspace
Current schema: default


In [0]:
#Now Read CSV
from pyspark.sql import functions as F

base_path = "/Volumes/clickstream_lab/default/raw_files"

files = [
    "batch1.csv",
    "batch2_advance_watermark.csv",
    "batch3_finalize_sessions.csv",
    "batch4_late_events.csv"
]

for file in files:
    

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}/{file}")
    )

print("Columns and data types:")
print("First 5 records:")
df.printSchema()
df.limit(5).display()   

Columns and data types:
First 5 records:
root
 |-- event_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- page: string (nullable = true)
 |-- action: string (nullable = true)



event_id,user_id,event_time,page,action
14,u1,2026-10-13T10:02:00.000Z,/payment,purchase
15,u2,2026-10-13T10:03:00.000Z,/checkout,purchase
16,u3,2026-10-13T10:04:00.000Z,/products/late,click


In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/raw_files"

batch1 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch1.csv")
batch2 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch2.csv")
batch3 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch3.csv")
batch4 = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/batch4.csv")

print("Batch 1:", batch1.count(), "rows")
print("Batch 2:", batch2.count(), "rows")
print("Batch 3:", batch3.count(), "rows")
print("Batch 4:", batch4.count(), "rows")

print("Batch 1 columns:", batch1.columns)

Batch 1: 5 rows
Batch 2: 5 rows
Batch 3: 5 rows
Batch 4: 5 rows
Batch 1 columns: ['user_id', 'event_time', 'event_type', 'page_url', 'device']


In [0]:
# Now we combine batch1 to batch 4
from pyspark.sql import functions as F

all_events = (
    batch1
    .unionByName(batch2)
    .unionByName(batch3)
    .unionByName(batch4)
    .withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )
)

display(all_events.orderBy("event_time"))


user_id,event_time,event_type,page_url,device
1,2026-01-01T10:00:00.000Z,page_view,/home,desktop
1,2026-01-01T10:05:00.000Z,click,/products,desktop
1,2026-01-01T10:10:00.000Z,page_view,/product/123,desktop
2,2026-01-01T11:00:00.000Z,page_view,/home,mobile
2,2026-01-01T11:02:00.000Z,scroll,/blog/post1,mobile
3,2026-01-01T12:00:00.000Z,page_view,/about,desktop
3,2026-01-01T12:03:00.000Z,click,/contact,desktop
4,2026-01-01T13:00:00.000Z,page_view,/home,tablet
4,2026-01-01T13:15:00.000Z,page_view,/products,tablet
5,2026-01-01T14:00:00.000Z,click,/signup,mobile


In [0]:
# Now we check event timestamp
all_events.select(
    "user_id",
    "event_time",
    "event_type",
    "page_url",
    "device"
).orderBy("event_time").show(30, truncate=False)

+-------+-------------------+----------+------------+-------+
|user_id|event_time         |event_type|page_url    |device |
+-------+-------------------+----------+------------+-------+
|1      |2026-01-01 10:00:00|page_view |/home       |desktop|
|1      |2026-01-01 10:05:00|click     |/products   |desktop|
|1      |2026-01-01 10:10:00|page_view |/product/123|desktop|
|2      |2026-01-01 11:00:00|page_view |/home       |mobile |
|2      |2026-01-01 11:02:00|scroll    |/blog/post1 |mobile |
|3      |2026-01-01 12:00:00|page_view |/about      |desktop|
|3      |2026-01-01 12:03:00|click     |/contact    |desktop|
|4      |2026-01-01 13:00:00|page_view |/home       |tablet |
|4      |2026-01-01 13:15:00|page_view |/products   |tablet |
|5      |2026-01-01 14:00:00|click     |/signup     |mobile |
|1      |2026-01-02 09:00:00|page_view |/home       |desktop|
|1      |2026-01-02 09:05:00|click     |/cart       |desktop|
|2      |2026-01-02 10:00:00|page_view |/products   |mobile |
|3      

In [0]:
#Sessionization with watermark
from pyspark.sql import functions as F

sessionized_df = (
    all_events
    .withWatermark("event_time", "10 minutes") # it manage the late event time state
    .groupBy(
        "user_id",
        F.session_window("event_time", "10 minutes") #
    )
    .agg(
        F.count("*").alias("event_count") # it count the event
    )
    .select(
        "user_id",
        F.col("session_window.start").alias("session_start"),
        F.col("session_window.end").alias("session_end"),
        "event_count"
    )
)

display(sessionized_df)

user_id,session_start,session_end,event_count
1,2026-01-01T10:00:00.000Z,2026-01-01T10:20:00.000Z,3
1,2026-01-02T09:00:00.000Z,2026-01-02T09:15:00.000Z,2
2,2026-01-01T11:00:00.000Z,2026-01-01T11:12:00.000Z,2
2,2026-01-02T10:00:00.000Z,2026-01-02T10:10:00.000Z,1
3,2026-01-01T12:00:00.000Z,2026-01-01T12:13:00.000Z,2
3,2026-01-02T11:00:00.000Z,2026-01-02T11:20:00.000Z,2
4,2026-01-01T13:00:00.000Z,2026-01-01T13:10:00.000Z,1
4,2026-01-01T13:15:00.000Z,2026-01-01T13:25:00.000Z,1
4,2026-01-02T12:00:00.000Z,2026-01-02T12:10:00.000Z,1
5,2026-01-01T14:00:00.000Z,2026-01-01T14:10:00.000Z,1


In [0]:
# Now we write session result to silver
silver_path = "/Volumes/clickstream_lab/default/raw_files/silver_sessions"

(
    sessionized_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("Sessionized results saved to Silver Delta storage.")

Sessionized results saved to Silver Delta storage.


In [0]:
# Now we verfiy the silver result 
silver_df = spark.read.format("delta").load(silver_path)

display(silver_df.orderBy("user_id", "session_start"))

user_id,session_start,session_end,event_count
1,2026-01-01T10:00:00.000Z,2026-01-01T10:20:00.000Z,3
1,2026-01-02T09:00:00.000Z,2026-01-02T09:15:00.000Z,2
2,2026-01-01T11:00:00.000Z,2026-01-01T11:12:00.000Z,2
2,2026-01-02T10:00:00.000Z,2026-01-02T10:10:00.000Z,1
3,2026-01-01T12:00:00.000Z,2026-01-01T12:13:00.000Z,2
3,2026-01-02T11:00:00.000Z,2026-01-02T11:20:00.000Z,2
4,2026-01-01T13:00:00.000Z,2026-01-01T13:10:00.000Z,1
4,2026-01-01T13:15:00.000Z,2026-01-01T13:25:00.000Z,1
4,2026-01-02T12:00:00.000Z,2026-01-02T12:10:00.000Z,1
5,2026-01-01T14:00:00.000Z,2026-01-01T14:10:00.000Z,1


In [0]:
all_events.select(
    F.min("event_time").alias("earliest_event"),
    F.max("event_time").alias("latest_event"),
    F.count("*").alias("total_events")
).show(truncate=False)

+-------------------+-------------------+------------+
|earliest_event     |latest_event       |total_events|
+-------------------+-------------------+------------+
|2026-01-01 10:00:00|2026-01-02 14:20:00|20          |
+-------------------+-------------------+------------+



In [0]:
sessionized_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("clickstream_lab.default.clickstream_silver")

In [0]:
display(
    spark.table("clickstream_lab.default.clickstream_silver")
)

user_id,session_start,session_end,event_count
1,2026-01-01T10:00:00.000Z,2026-01-01T10:20:00.000Z,3
1,2026-01-02T09:00:00.000Z,2026-01-02T09:15:00.000Z,2
2,2026-01-01T11:00:00.000Z,2026-01-01T11:12:00.000Z,2
2,2026-01-02T10:00:00.000Z,2026-01-02T10:10:00.000Z,1
3,2026-01-01T12:00:00.000Z,2026-01-01T12:13:00.000Z,2
3,2026-01-02T11:00:00.000Z,2026-01-02T11:20:00.000Z,2
4,2026-01-01T13:00:00.000Z,2026-01-01T13:10:00.000Z,1
4,2026-01-01T13:15:00.000Z,2026-01-01T13:25:00.000Z,1
4,2026-01-02T12:00:00.000Z,2026-01-02T12:10:00.000Z,1
5,2026-01-01T14:00:00.000Z,2026-01-01T14:10:00.000Z,1


In [0]:
# verify the silver table
display(
    spark.table("clickstream_lab.default.clickstream_silver")
)

user_id,session_start,session_end,event_count
1,2026-01-01T10:00:00.000Z,2026-01-01T10:20:00.000Z,3
1,2026-01-02T09:00:00.000Z,2026-01-02T09:15:00.000Z,2
2,2026-01-01T11:00:00.000Z,2026-01-01T11:12:00.000Z,2
2,2026-01-02T10:00:00.000Z,2026-01-02T10:10:00.000Z,1
3,2026-01-01T12:00:00.000Z,2026-01-01T12:13:00.000Z,2
3,2026-01-02T11:00:00.000Z,2026-01-02T11:20:00.000Z,2
4,2026-01-01T13:00:00.000Z,2026-01-01T13:10:00.000Z,1
4,2026-01-01T13:15:00.000Z,2026-01-01T13:25:00.000Z,1
4,2026-01-02T12:00:00.000Z,2026-01-02T12:10:00.000Z,1
5,2026-01-01T14:00:00.000Z,2026-01-01T14:10:00.000Z,1


In [0]:
# Now we create streming query for the late event run 
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/raw_files"

stream_df = (
    spark.readStream
    .schema(batch1.schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .csv(base_path)
    .withColumn("event_time", F.to_timestamp("event_time"))
)

sessionized_stream = (
    stream_df
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        "user_id",
        F.session_window("event_time", "10 minutes")
    )
    .agg(F.count("*").alias("event_count"))
    .select(
        "user_id",
        F.col("session_window.start").alias("session_start"),
        F.col("session_window.end").alias("session_end"),
        "event_count"
    )
)

In [0]:
#start the stream
query = (
    sessionized_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/raw_files/checkpoints/sessionization_test"
    )
    .trigger(availableNow=True)
    .toTable("clickstream_lab.default.clickstream_silver")
)

In [0]:
#steam-static joins
from pyspark.sql import functions as F

device_lookup = spark.createDataFrame(
    [
        ("mobile", "Mobile Device"),
        ("desktop", "Desktop Computer"),
        ("tablet", "Tablet Device")
    ],
    ["device", "device_description"]
)

enriched_events = batch1.join(
    device_lookup,
    on="device",
    how="left"
)

display(enriched_events)

device,user_id,event_time,event_type,page_url,device_description
desktop,1,2026-01-01T10:00:00.000Z,page_view,/home,Desktop Computer
desktop,1,2026-01-01T10:05:00.000Z,click,/products,Desktop Computer
desktop,1,2026-01-01T10:10:00.000Z,page_view,/product/123,Desktop Computer
mobile,2,2026-01-01T11:00:00.000Z,page_view,/home,Mobile Device
mobile,2,2026-01-01T11:02:00.000Z,scroll,/blog/post1,Mobile Device


In [0]:
# Save the silver in delta table
(
    sessionized_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("clickstream_lab.default.clickstream_silver")
)

In [0]:
#Veify it save or not
display(
    spark.table("clickstream_lab.default.clickstream_silver")
)

user_id,session_start,session_end,event_count
1,2026-01-01T10:00:00.000Z,2026-01-01T10:20:00.000Z,3
1,2026-01-02T09:00:00.000Z,2026-01-02T09:15:00.000Z,2
2,2026-01-01T11:00:00.000Z,2026-01-01T11:12:00.000Z,2
2,2026-01-02T10:00:00.000Z,2026-01-02T10:10:00.000Z,1
3,2026-01-01T12:00:00.000Z,2026-01-01T12:13:00.000Z,2
3,2026-01-02T11:00:00.000Z,2026-01-02T11:20:00.000Z,2
4,2026-01-01T13:00:00.000Z,2026-01-01T13:10:00.000Z,1
4,2026-01-01T13:15:00.000Z,2026-01-01T13:25:00.000Z,1
4,2026-01-02T12:00:00.000Z,2026-01-02T12:10:00.000Z,1
5,2026-01-01T14:00:00.000Z,2026-01-01T14:10:00.000Z,1


In [0]:
# Check streaming query status
print("Streaming query active:", query.isActive)
print("Streaming query status:", query.status)

Streaming query active: False
Streaming query status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
